# 임베딩 입력 방식 비교 테스트

동일한 쿼리로 3가지 임베딩 입력 방식의 검색 품질을 비교합니다.

| 방식 | 임베딩 입력 | 특징 |
|---|---|---|
| A. PDF 바이트 | 1페이지 PDF 통째로 | 이미지+텍스트+레이아웃 |
| B. 텍스트만 | `page.get_text()` | 순수 텍스트 |
| C. 페이지 이미지 | 페이지를 PNG로 렌더링 | 시각 정보 보존 |

In [10]:
import os
import time
import io
import numpy as np
import faiss
import fitz
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

EMBEDDING_MODEL = "gemini-embedding-2-preview"
EMBEDDING_DIM = 3072
PDF_PATH = "data/헤리티지 역사와 과학 제58권 제4호(통권 제110권).pdf"

test_queries = [
    "어떤 연구들을 진행했었는가?",
    "탕건에 대해서 과학적 분석 결과가 어떻게 되었는가?",
    "두 자락치마의 조선 왕족 실록의 기록은?",
    "훈장 가운데에 있는 명칭은?",
]

print("설정 완료")

설정 완료


## 3가지 방식으로 페이지 데이터 준비

In [11]:
# 3가지 방식으로 페이지 데이터 준비
doc = fitz.open(PDF_PATH)
total_pages = len(doc)

# A. PDF 바이트 (1페이지짜리 PDF)
page_pdfs = []
for i in range(total_pages):
    single = fitz.open()
    single.insert_pdf(doc, from_page=i, to_page=i)
    page_pdfs.append(single.tobytes())
    single.close()

# B. 텍스트만
page_texts = []
for i in range(total_pages):
    text = doc[i].get_text().strip()
    page_texts.append(text if text else " ")  # 빈 페이지 방지

# C. 페이지 이미지 (PNG)
page_images = []
for i in range(total_pages):
    mat = fitz.Matrix(150 / 72, 150 / 72)
    pix = doc[i].get_pixmap(matrix=mat)
    page_images.append(pix.tobytes("png"))

doc.close()
print(f"총 {total_pages}페이지 준비 완료")
print(f"  A. PDF 바이트: 첫 페이지 {len(page_pdfs[0])/1024:.0f}KB")
print(f"  B. 텍스트: 첫 페이지 {len(page_texts[0])}자")
print(f"  C. 이미지: 첫 페이지 {len(page_images[0])/1024:.0f}KB")

총 260페이지 준비 완료
  A. PDF 바이트: 첫 페이지 93KB
  B. 텍스트: 첫 페이지 178자
  C. 이미지: 첫 페이지 331KB


## 임베딩 생성 (3가지 방식)

In [3]:
# 공통 임베딩 함수
def embed_batch(contents_list: list, batch_size: int = 5, label: str = "") -> np.ndarray:
    """contents 리스트를 배치로 임베딩"""
    all_embeddings = []
    total = len(contents_list)
    t0 = time.time()

    for i in range(0, total, batch_size):
        batch = contents_list[i : i + batch_size]
        response = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(
                output_dimensionality=EMBEDDING_DIM,
                task_type="RETRIEVAL_DOCUMENT",
            ),
        )
        for emb in response.embeddings:
            all_embeddings.append(emb.values)
        if (i // batch_size) % 10 == 0:
            print(f"  [{label}] {min(i+batch_size, total)}/{total}")

    elapsed = time.time() - t0
    arr = np.array(all_embeddings, dtype=np.float32)
    print(f"  [{label}] 완료: {arr.shape} ({elapsed:.1f}초)")
    return arr

def build_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    """FAISS 인덱스 구축"""
    emb = embeddings.copy()
    faiss.normalize_L2(emb)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(emb)
    return index

In [4]:
# A. PDF 바이트 임베딩
print("=== A. PDF 바이트 임베딩 ===")
contents_a = [
    types.Content(parts=[types.Part.from_bytes(data=pdf, mime_type="application/pdf")])
    for pdf in page_pdfs
]
emb_a = embed_batch(contents_a, label="PDF")
index_a = build_index(emb_a)

=== A. PDF 바이트 임베딩 ===
  [PDF] 5/260
  [PDF] 55/260
  [PDF] 105/260
  [PDF] 155/260
  [PDF] 205/260
  [PDF] 255/260
  [PDF] 완료: (260, 3072) (118.3초)


In [5]:
# B. 텍스트만 임베딩
print("=== B. 텍스트 임베딩 ===")
emb_b = embed_batch(page_texts, label="텍스트")
index_b = build_index(emb_b)

=== B. 텍스트 임베딩 ===
  [텍스트] 5/260
  [텍스트] 55/260
  [텍스트] 105/260
  [텍스트] 155/260
  [텍스트] 205/260
  [텍스트] 255/260
  [텍스트] 완료: (260, 3072) (37.0초)


In [6]:
# C. 페이지 이미지 임베딩
print("=== C. 이미지 임베딩 ===")
contents_c = [
    types.Content(parts=[types.Part.from_bytes(data=img, mime_type="image/png")])
    for img in page_images
]
emb_c = embed_batch(contents_c, label="이미지")
index_c = build_index(emb_c)

=== C. 이미지 임베딩 ===
  [이미지] 5/260
  [이미지] 55/260
  [이미지] 105/260
  [이미지] 155/260
  [이미지] 205/260
  [이미지] 255/260
  [이미지] 완료: (260, 3072) (76.7초)


## 비교 검색 실행

In [12]:
# 쿼리 임베딩 → 3개 인덱스에서 각각 검색
def search_index(index: faiss.IndexFlatIP, query_emb: np.ndarray, top_k: int = 5):
    """인덱스에서 검색하여 (페이지, 유사도) 리스트 반환"""
    scores, indices = index.search(query_emb, top_k)
    return [(int(idx) + 1, float(score)) for idx, score in zip(indices[0], scores[0])]

def embed_query(query: str) -> np.ndarray:
    """쿼리 텍스트 임베딩"""
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(
            output_dimensionality=EMBEDDING_DIM,
            task_type="RETRIEVAL_QUERY",
        ),
    )
    emb = np.array([response.embeddings[0].values], dtype=np.float32)
    faiss.normalize_L2(emb)
    return emb

# 비교 실행
for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*80}")
    print(f"쿼리 {i}: {query}")
    print(f"{'='*80}")

    query_emb = embed_query(query)

    results_a = search_index(index_a, query_emb)
    results_b = search_index(index_b, query_emb)
    results_c = search_index(index_c, query_emb)

    print(f"\n{'방식':<12} | {'#1':>20} | {'#2':>20} | {'#3':>20} | {'#4':>20} | {'#5':>20}")
    print(f"{'':-<12}-+-{'':->20}-+-{'':->20}-+-{'':->20}-+-{'':->20}-+-{'':->20}")

    for label, results in [("A. PDF", results_a), ("B. 텍스트", results_b), ("C. 이미지", results_c)]:
        cols = [f"p{page}({score:.4f})" for page, score in results]
        print(f"{label:<12} | {cols[0]:>20} | {cols[1]:>20} | {cols[2]:>20} | {cols[3]:>20} | {cols[4]:>20}")

    # 3가지 방식의 top-1 유사도 비교
    print(f"\n  Top-1 유사도: A={results_a[0][1]:.4f}  B={results_b[0][1]:.4f}  C={results_c[0][1]:.4f}")
    
    # 페이지 겹침 분석
    pages_a = set(p for p, _ in results_a)
    pages_b = set(p for p, _ in results_b)
    pages_c = set(p for p, _ in results_c)
    ab = pages_a & pages_b
    ac = pages_a & pages_c
    bc = pages_b & pages_c
    abc = pages_a & pages_b & pages_c
    print(f"  페이지 겹침: A∩B={len(ab)} A∩C={len(ac)} B∩C={len(bc)} A∩B∩C={len(abc)}")


쿼리 1: 어떤 연구들을 진행했었는가?

방식           |                   #1 |                   #2 |                   #3 |                   #4 |                   #5
-------------+----------------------+----------------------+----------------------+----------------------+---------------------
A. PDF       |          p90(0.4178) |         p129(0.4067) |         p104(0.4016) |         p130(0.4011) |         p243(0.3998)
B. 텍스트       |          p95(0.6367) |         p107(0.6277) |         p129(0.6256) |         p145(0.6237) |         p226(0.6220)
C. 이미지       |         p229(0.4076) |         p130(0.4041) |         p129(0.4030) |          p43(0.3992) |         p104(0.3990)

  Top-1 유사도: A=0.4178  B=0.6367  C=0.4076
  페이지 겹침: A∩B=1 A∩C=3 B∩C=1 A∩B∩C=1

쿼리 2: 탕건에 대해서 과학적 분석 결과가 어떻게 되었는가?

방식           |                   #1 |                   #2 |                   #3 |                   #4 |                   #5
-------------+----------------------+----------------------+----------------------+---------

## 종합 요약

In [9]:
# 전체 쿼리에 대한 평균 유사도 비교
print("="*60)
print("전체 쿼리 평균 Top-1/Top-3/Top-5 유사도")
print("="*60)

for label, idx in [("A. PDF 바이트", index_a), ("B. 텍스트만", index_b), ("C. 페이지 이미지", index_c)]:
    top1_scores = []
    top3_scores = []
    top5_scores = []
    for query in test_queries:
        query_emb = embed_query(query)
        results = search_index(idx, query_emb)
        top1_scores.append(results[0][1])
        top3_scores.append(np.mean([s for _, s in results[:3]]))
        top5_scores.append(np.mean([s for _, s in results[:5]]))

    print(f"\n  {label}")
    print(f"    Top-1 평균: {np.mean(top1_scores):.4f}")
    print(f"    Top-3 평균: {np.mean(top3_scores):.4f}")
    print(f"    Top-5 평균: {np.mean(top5_scores):.4f}")

전체 쿼리 평균 Top-1/Top-3/Top-5 유사도

  A. PDF 바이트
    Top-1 평균: 0.4366
    Top-3 평균: 0.4318
    Top-5 평균: 0.4289

  B. 텍스트만
    Top-1 평균: 0.6441
    Top-3 평균: 0.6369
    Top-5 평균: 0.6331

  C. 페이지 이미지
    Top-1 평균: 0.4499
    Top-3 평균: 0.4384
    Top-5 평균: 0.4329
